In [6]:
import pandas as pd, numpy as np

# === 路徑 ===
a_path = 'ds_doService_combo_enhanced.csv'   # service features
b_path = 'ds_billing_onehot.csv'     # Billing features
c_path = 'csr_analysis\\final_output\\enhanced_customer_features_with_analysis.csv'   # CSR features
d_path = 'cleaned_dataset2.csv'                    # dataset for final status at 2024/12/31
out_path = 'csr_service_bill_SFSS.csv'

# === A, B ===
a = pd.read_csv(a_path, sep='^').rename(columns={'客編': 'CUST_NO'})
b = pd.read_csv(b_path, sep=',').rename(columns={'客編': 'CUST_NO'})
c = pd.read_csv(c_path, sep=',')
d = pd.read_csv(d_path, sep='^').rename(columns={'客編': 'CUST_NO'})


In [28]:
d.head()

,客編,產品名稱,用戶種類,相關編號,起日,迄日,系統台,地區,繳別,使用狀態
0,503,EPON,一般收視戶,2569881,2023/04/20 11:17:58,NaN,大屯,大里區,1,使用中
1,506,CATV,一般收視戶,709,1998/11/01 00:00:00,1998/11/30 00:00:00,大屯,太平區,2,停用
2,511,CATV,一般收視戶,714,2005/07/19 15:50:03,NaN,大屯,太平區,1,停用
3,511,CATV,一般收視戶,713,1997/06/01 00:00:00,1997/08/31 00:00:00,大屯,太平區,3,停用
4,511,DTV,一般收視戶,1262815,2014/06/27 19:07:39,NaN,大屯,太平區,1,停用


In [16]:
print("=== 檢查 a、b、c 三個資料集中 DType 為 object 的欄位 ===")

# 檢查 a 資料集
print("\n=== a 資料集 (工單特徵) ===")
print(f"a 資料集形狀: {a.shape}")
a_object_cols = a.select_dtypes(include=['object']).columns.tolist()
print(f"object 類型欄位數量: {len(a_object_cols)}")
if a_object_cols:
    print("object 類型欄位清單:")
    for i, col in enumerate(a_object_cols, 1):
        print(f"  {i:2d}. {col}")
        # 顯示該欄位的前幾個值作為範例
        sample_values = a[col].dropna().unique()[:3]
        print(f"      範例值: {list(sample_values)}")
else:
    print("  ✅ 沒有 object 類型欄位")

# 檢查 b 資料集
print("\n=== b 資料集 (帳單資料特徵) ===")
print(f"b 資料集形狀: {b.shape}")
b_object_cols = b.select_dtypes(include=['object']).columns.tolist()
print(f"object 類型欄位數量: {len(b_object_cols)}")
if b_object_cols:
    print("object 類型欄位清單:")
    for i, col in enumerate(b_object_cols, 1):
        print(f"  {i:2d}. {col}")
        # 顯示該欄位的前幾個值作為範例
        sample_values = b[col].dropna().unique()[:3]
        print(f"      範例值: {list(sample_values)}")
else:
    print("  ✅ 沒有 object 類型欄位")

# 檢查 c 資料集
print("\n=== c 資料集 (來電記錄特徵) ===")
print(f"c 資料集形狀: {c.shape}")
c_object_cols = c.select_dtypes(include=['object']).columns.tolist()
print(f"object 類型欄位數量: {len(c_object_cols)}")
if c_object_cols:
    print("object 類型欄位清單:")
    for i, col in enumerate(c_object_cols, 1):
        print(f"  {i:2d}. {col}")
        # 顯示該欄位的前幾個值作為範例
        sample_values = c[col].dropna().unique()[:3]
        print(f"      範例值: {list(sample_values)}")
else:
    print("  ✅ 沒有 object 類型欄位")

# 總結
print(f"\n=== 總結 ===")
total_object_cols = len(a_object_cols) + len(b_object_cols) + len(c_object_cols)
print(f"a 資料集 object 欄位: {len(a_object_cols)} 個")
print(f"b 資料集 object 欄位: {len(b_object_cols)} 個")
print(f"c 資料集 object 欄位: {len(c_object_cols)} 個")
print(f"總計 object 欄位: {total_object_cols} 個")

# 如果有 object 欄位，建議處理方式
if total_object_cols > 0:
    print(f"\n=== 建議處理方式 ===")
    print("1. 檢查是否為類別變數 → 使用 pd.get_dummies() 進行 one-hot 編碼")
    print("2. 檢查是否為日期時間 → 使用 pd.to_datetime() 轉換")
    print("3. 檢查是否為數值但儲存為文字 → 使用 pd.to_numeric() 轉換")
    print("4. 檢查是否包含空值或特殊字符 → 進行資料清理")
else:
    print(f"\n✅ 三個資料集都沒有 object 類型欄位，資料類型處理良好")

# 詳細資料型態分析
print(f"\n=== 詳細資料型態分析 ===")
for dataset_name, dataset in [('a', a), ('b', b), ('c', c)]:
    print(f"\n{dataset_name} 資料集資料型態統計:")
    dtype_counts = dataset.dtypes.value_counts()
    for dtype, count in dtype_counts.items():
        print(f"  {dtype}: {count} 個欄位")

=== 檢查 a、b、c 三個資料集中 DType 為 object 的欄位 ===

=== a 資料集 (工單特徵) ===
a 資料集形狀: (49871, 44)
object 類型欄位數量: 9
object 類型欄位清單:
   1. 工單原因_主要
      範例值: ['用戶端問題_用戶不善操作', '用戶端問題_IP分享(集線)器,無線AP問題', '其他_測試正常']
   2. 工單原因_次要
      範例值: ['tinp_CMTS問題', '用戶端問題_IP分享(集線)器,無線AP問題', '用戶端問題_數據機故,更換']
   3. 工單原因_最新
      範例值: ['其他_測試正常', '用戶端問題_IP分享(集線)器,無線AP問題', 'RF線路問題_更換室外5C頭']
   4. 工單原因_最舊
      範例值: ['用戶端問題_用戶不善操作', '用戶端問題_IP分享(集線)器,無線AP問題', '其他_測試正常']
   5. 工單原因_文字特徵
      範例值: ['用戶端問題_用戶不善操作 | tinp_CMTS問題 | 用戶端問題_IP分享(集線)器,無線AP問題 | 其他_測試正常', '用戶端問題_IP分享(集線)器,無線AP問題', '其他_測試正常']
   6. 產品名稱
      範例值: ['EPON', 'CM']
   7. 使用狀態
      範例值: ['使用中', '停用', '欠款斷線']
   8. 最新工單日期
      範例值: ['2023-09-04 21:19:20', '2022-08-26 20:58:06', '2022-04-13 21:29:02']
   9. 工單日期
      範例值: ['2023-09-04 21:19:20', '2022-08-26 20:58:06', '2022-04-13 21:29:02']

=== b 資料集 (帳單資料特徵) ===
b 資料集形狀: (19289, 16)
object 類型欄位數量: 0
  ✅ 沒有 object 類型欄位

=== c 資料集 (來電記錄特徵) ===
c 資料集形狀: (44349, 144)
object 類型欄位數量: 1
object 類型欄位清單:
   

In [19]:
d.head()

,客編,產品名稱,用戶種類,相關編號,起日,迄日,系統台,地區,繳別,使用狀態
0,503,EPON,一般收視戶,2569881,2023/04/20 11:17:58,NaN,大屯,大里區,1,使用中
1,506,CATV,一般收視戶,709,1998/11/01 00:00:00,1998/11/30 00:00:00,大屯,太平區,2,停用
2,511,CATV,一般收視戶,714,2005/07/19 15:50:03,NaN,大屯,太平區,1,停用
3,511,CATV,一般收視戶,713,1997/06/01 00:00:00,1997/08/31 00:00:00,大屯,太平區,3,停用
4,511,DTV,一般收視戶,1262815,2014/06/27 19:07:39,NaN,大屯,太平區,1,停用


In [7]:
# 處理d資料集：篩選EPON/CM產品並處理重複客編
print("=== 處理d資料集：篩選EPON/CM產品並處理重複客編 ===")

# 步驟1: 篩選產品名稱為EPON或CM的資料
print("=== 步驟1: 篩選產品名稱 ===")
print(f"原始d資料集形狀: {d.shape}")
print("原始產品名稱分布:")
print(d['產品名稱'].value_counts().head(10))

# 篩選僅保留EPON或CM的資料
d_filtered = d[d['產品名稱'].isin(['EPON', 'CM'])].copy()

print(f"\n篩選後結果:")
print(f"篩選前筆數: {len(d)}")
print(f"篩選後筆數: {len(d_filtered)}")
print(f"刪除筆數: {len(d) - len(d_filtered)}")
print(f"保留比例: {len(d_filtered) / len(d) * 100:.1f}%")

print(f"\n篩選後產品名稱分布:")
print(d_filtered['產品名稱'].value_counts())

# 步驟2: 檢查重複客編狀況
print(f"\n=== 步驟2: 檢查重複客編狀況 ===")
total_records = len(d_filtered)
unique_customers = d_filtered['CUST_NO'].nunique()
duplicate_count = total_records - unique_customers

print(f"總記錄數: {total_records:,}")
print(f"唯一客編數: {unique_customers:,}")
print(f"重複記錄數: {duplicate_count:,}")
print(f"重複比例: {duplicate_count / total_records * 100:.2f}%")

if duplicate_count > 0:
    # 找出有重複的客編
    customer_counts = d_filtered['CUST_NO'].value_counts()
    duplicated_customers = customer_counts[customer_counts > 1]
    
    print(f"\n有重複記錄的客編數量: {len(duplicated_customers)}")
    print(f"重複次數分布:")
    duplicate_distribution = duplicated_customers.value_counts().sort_index()
    for count, freq in duplicate_distribution.items():
        print(f"  重複{count}次的客編: {freq} 個")
    
    # 顯示重複客編範例（處理前）
    print(f"\n=== 重複客編範例（處理前） ===")
    sample_customers = duplicated_customers.head(3).index
    for cust_no in sample_customers:
        records = d_filtered[d_filtered['CUST_NO'] == cust_no].sort_values('起日')
        print(f"\n客編 {cust_no} 的記錄:")
        for i, (idx, row) in enumerate(records.iterrows(), 1):
            print(f"  記錄{i}: 起日={row['起日']}, 迄日={row['迄日']}, 使用狀態={row['使用狀態']}")

# 步驟3: 處理重複客編 - 修改後的邏輯
print(f"\n=== 步驟3: 處理重複客編 - 修改後的邏輯 ===")
print("處理邏輯:")
print("1. 迄日為空值時保留有空值的記錄")
print("2. 迄日均無空值者保留起日最晚的記錄")

# 檢查起日和迄日欄位的資料型態
print(f"\n起日欄位資料型態: {d_filtered['起日'].dtype}")
print(f"迄日欄位資料型態: {d_filtered['迄日'].dtype}")
print(f"起日範例值: {d_filtered['起日'].head().tolist()}")
print(f"迄日範例值: {d_filtered['迄日'].head().tolist()}")

# 檢查迄日空值情況
null_end_dates = d_filtered['迄日'].isnull().sum()
print(f"迄日空值數量: {null_end_dates}")

# 轉換起日為日期格式（如果需要）
if d_filtered['起日'].dtype == 'object':
    print("\n轉換起日為日期格式...")
    try:
        d_filtered['起日_date'] = pd.to_datetime(d_filtered['起日'], errors='coerce')
    except:
        d_filtered['起日_date'] = pd.to_datetime(d_filtered['起日'], format='%Y%m%d', errors='coerce')
    
    null_dates = d_filtered['起日_date'].isnull().sum()
    if null_dates > 0:
        print(f"⚠️ 有 {null_dates} 筆起日無法轉換為日期格式")
    else:
        print("✅ 所有起日都成功轉換為日期格式")
else:
    d_filtered['起日_date'] = d_filtered['起日']

print(f"\n執行去重處理...")

# 定義去重函數
def deduplicate_records(group):
    """
    對每個客編的重複記錄進行去重
    1. 如果有迄日為空值的記錄，優先保留迄日為空值的記錄
    2. 如果迄日都不為空值，保留起日最晚的記錄
    """
    if len(group) == 1:
        # 只有一筆記錄，直接返回
        return group
    
    # 檢查是否有迄日為空值的記錄
    null_end_records = group[group['迄日'].isnull()]
    
    if len(null_end_records) > 0:
        # 有迄日為空值的記錄，保留其中一筆（如果有多筆空值記錄，取起日最晚的）
        if len(null_end_records) == 1:
            return null_end_records
        else:
            # 多筆迄日為空值記錄，取起日最晚的
            return null_end_records.loc[null_end_records['起日_date'].idxmax():null_end_records['起日_date'].idxmax()]
    else:
        # 迄日都不為空值，保留起日最晚的記錄
        return group.loc[group['起日_date'].idxmax():group['起日_date'].idxmax()]

# 執行去重
print("正在執行去重處理...")
d_deduped_list = []

for cust_no, group in d_filtered.groupby('CUST_NO'):
    deduped_record = deduplicate_records(group)
    d_deduped_list.append(deduped_record)

d_deduped = pd.concat(d_deduped_list, ignore_index=True)

print(f"\n=== 去重結果 ===")
print(f"去重前筆數: {len(d_filtered)}")
print(f"去重後筆數: {len(d_deduped)}")
print(f"刪除筆數: {len(d_filtered) - len(d_deduped)}")
print(f"保留比例: {len(d_deduped) / len(d_filtered) * 100:.2f}%")

# 驗證去重結果
remaining_duplicates = d_deduped['CUST_NO'].duplicated().sum()
print(f"去重後剩餘重複客編數: {remaining_duplicates}")

if remaining_duplicates == 0:
    print("✅ 所有重複客編都已處理完成")
else:
    print("⚠️ 仍有重複客編存在")

# 分析去重策略的使用情況
if duplicate_count > 0:
    print(f"\n=== 去重策略分析 ===")
    
    # 統計保留記錄的迄日狀況
    retained_null_end = d_deduped['迄日'].isnull().sum()
    retained_non_null_end = len(d_deduped) - retained_null_end
    
    print(f"保留記錄中迄日為空值的數量: {retained_null_end}")
    print(f"保留記錄中迄日非空值的數量: {retained_non_null_end}")
    
    # 對於有重複記錄的客編，檢查應用的策略
    strategy_analysis = []
    
    for cust_no in duplicated_customers.index:
        original_records = d_filtered[d_filtered['CUST_NO'] == cust_no]
        retained_record = d_deduped[d_deduped['CUST_NO'] == cust_no]
        
        if len(retained_record) > 0:
            retained_record = retained_record.iloc[0]
            null_end_count = original_records['迄日'].isnull().sum()
            
            if null_end_count > 0 and pd.isnull(retained_record['迄日']):
                strategy_analysis.append('保留迄日空值記錄')
            elif null_end_count == 0:
                strategy_analysis.append('保留起日最晚記錄')
            else:
                strategy_analysis.append('其他情況')
    
    strategy_counts = pd.Series(strategy_analysis).value_counts()
    print(f"\n去重策略統計:")
    for strategy, count in strategy_counts.items():
        print(f"  {strategy}: {count} 個客編")

# 顯示處理後的範例
if duplicate_count > 0:
    print(f"\n=== 重複客編範例（處理後） ===")
    for cust_no in sample_customers:
        # 原始記錄
        original_records = d_filtered[d_filtered['CUST_NO'] == cust_no]
        retained_record = d_deduped[d_deduped['CUST_NO'] == cust_no]
        
        if len(retained_record) > 0:
            retained_record = retained_record.iloc[0]
            print(f"\n客編 {cust_no}:")
            print(f"  原始記錄數: {len(original_records)}")
            print(f"  保留記錄: 起日={retained_record['起日']}, 迄日={retained_record['迄日']}, 使用狀態={retained_record['使用狀態']}")
            
            # 判斷使用的策略
            null_end_count = original_records['迄日'].isnull().sum()
            if null_end_count > 0 and pd.isnull(retained_record['迄日']):
                print(f"  策略: 保留迄日空值記錄")
            elif null_end_count == 0:
                print(f"  策略: 保留起日最晚記錄")

# 清理臨時欄位
if '起日_date' in d_deduped.columns:
    d_deduped = d_deduped.drop(columns=['起日_date'])

# 更新d變數
d = d_deduped.copy()

print(f"\n✅ d資料集處理完成")
print(f"最終d資料集形狀: {d.shape}")
print(f"最終產品名稱分布:")
print(d['產品名稱'].value_counts())
print(f"最終使用狀態分布:")
print(d['使用狀態'].value_counts())
print(f"最終迄日空值數量: {d['迄日'].isnull().sum()}")

=== 處理d資料集：篩選EPON/CM產品並處理重複客編 ===
=== 步驟1: 篩選產品名稱 ===
原始d資料集形狀: (2124049, 10)
原始產品名稱分布:
產品名稱
DTV              888046
CATV             771864
CM               231239
EPON             110778
LINE TV          105253
熊搭心(瑪帛)            5374
LiTV               2822
HBOGO              2319
MESH WIFI加值服務      1752
居家智慧攝影機            1512
Name: count, dtype: int64

篩選後結果:
篩選前筆數: 2124049
篩選後筆數: 342017
刪除筆數: 1782032
保留比例: 16.1%

篩選後產品名稱分布:
產品名稱
CM      231239
EPON    110778
Name: count, dtype: int64

=== 步驟2: 檢查重複客編狀況 ===
總記錄數: 342,017
唯一客編數: 321,993
重複記錄數: 20,024
重複比例: 5.85%

有重複記錄的客編數量: 16858
重複次數分布:
  重複2次的客編: 15107 個
  重複3次的客編: 1186 個
  重複4次的客編: 170 個
  重複5次的客編: 249 個
  重複6次的客編: 39 個
  重複7次的客編: 51 個
  重複8次的客編: 10 個
  重複9次的客編: 22 個
  重複10次的客編: 4 個
  重複11次的客編: 5 個
  重複12次的客編: 3 個
  重複13次的客編: 1 個
  重複14次的客編: 2 個
  重複15次的客編: 5 個
  重複17次的客編: 3 個
  重複18次的客編: 1 個

=== 重複客編範例（處理前） ===

客編 1018357 的記錄:
  記錄1: 起日=2016/12/16 12:54:43, 迄日=nan, 使用狀態=使用中
  記錄2: 起日=2017/01/03 18:47:36, 迄日=2017/12/30 15:51:

In [8]:
# 針對d資料集處理使用狀態
print("=== 針對d資料集處理使用狀態 ===")

# 檢查處理前的使用狀態分布
print("=== 處理前使用狀態分布 ===")
print(f"處理前d資料集筆數: {len(d)}")
status_before = d['使用狀態'].value_counts()
print("處理前使用狀態分布:")
print(status_before)

# 定義狀態映射規則
print(f"\n=== 狀態處理規則 ===")
print("1. 改為停用: 欠款斷線、拆機中、退租(先斷線未拆機)")
print("2. 改為使用中: 維修中、設備加裝、派收中、暫停、移機中")
print("3. 刪除記錄: 無法完工、裝機中、促銷中，其他未列舉狀態")

# 定義要保留的狀態（會進行處理的狀態）
to_stop_using = ['欠款斷線', '拆機中', '退租(先斷線未拆機)']
to_using = ['維修中', '設備加裝', '派收中', '暫停', '移機中']
explicitly_delete = ['無法完工', '裝機中', '促銷中']
keep_as_is = ['使用中', '停用']  # 原本就是正確狀態的，保持不變

# 所有應該保留的狀態
all_keep_statuses = to_stop_using + to_using + keep_as_is

print(f"\n=== 狀態分析 ===")
print(f"將改為停用的狀態: {to_stop_using}")
print(f"將改為使用中的狀態: {to_using}")
print(f"明確要刪除的狀態: {explicitly_delete}")
print(f"保持不變的狀態: {keep_as_is}")

# 找出其他未列舉的狀態
current_statuses = set(d['使用狀態'].unique())
listed_statuses = set(to_stop_using + to_using + explicitly_delete + keep_as_is)
other_statuses = current_statuses - listed_statuses

print(f"\n=== 發現的其他未列舉狀態 ===")
if other_statuses:
    for status in sorted(other_statuses):
        count = (d['使用狀態'] == status).sum()
        print(f"  {status}: {count} 筆")
    print(f"其他未列舉狀態總計: {len(other_statuses)} 種")
else:
    print("  ✅ 沒有發現其他未列舉的狀態")

# 步驟1: 將指定狀態改為停用
print(f"\n=== 步驟1: 將指定狀態改為停用 ===")

existing_stop_statuses = []
for status in to_stop_using:
    count = (d['使用狀態'] == status).sum()
    if count > 0:
        existing_stop_statuses.append(status)
        print(f"  {status}: {count} 筆")
    else:
        print(f"  {status}: 0 筆 (不存在)")

# 執行狀態更改
if existing_stop_statuses:
    d.loc[d['使用狀態'].isin(existing_stop_statuses), '使用狀態'] = '停用'
    print(f"✅ 已將 {len(existing_stop_statuses)} 種狀態改為停用")
else:
    print("⚠️ 沒有找到需要改為停用的狀態")

# 步驟2: 將指定狀態改為使用中
print(f"\n=== 步驟2: 將指定狀態改為使用中 ===")

existing_using_statuses = []
for status in to_using:
    count = (d['使用狀態'] == status).sum()
    if count > 0:
        existing_using_statuses.append(status)
        print(f"  {status}: {count} 筆")
    else:
        print(f"  {status}: 0 筆 (不存在)")

# 執行狀態更改
if existing_using_statuses:
    d.loc[d['使用狀態'].isin(existing_using_statuses), '使用狀態'] = '使用中'
    print(f"✅ 已將 {len(existing_using_statuses)} 種狀態改為使用中")
else:
    print("⚠️ 沒有找到需要改為使用中的狀態")

# 步驟3: 刪除指定狀態和其他未列舉狀態的記錄
to_delete = explicitly_delete + list(other_statuses)
print(f"\n=== 步驟3: 刪除指定狀態和其他未列舉狀態的記錄 ===")

print("要刪除的狀態:")
total_delete_count = 0
for status in to_delete:
    count = (d['使用狀態'] == status).sum()
    total_delete_count += count
    if count > 0:
        print(f"  {status}: {count} 筆")

if len(other_statuses) > 0:
    print(f"\n其中包含 {len(other_statuses)} 種其他未列舉狀態")

print(f"總計需刪除: {total_delete_count} 筆")

# 執行記錄刪除
if total_delete_count > 0:
    d_before_delete = d.copy()
    d = d[~d['使用狀態'].isin(to_delete)].copy()
    
    actual_deleted = len(d_before_delete) - len(d)
    print(f"✅ 已刪除 {actual_deleted} 筆記錄")
    
    # 驗證刪除結果
    remaining_delete_statuses = d['使用狀態'].isin(to_delete).sum()
    if remaining_delete_statuses == 0:
        print("✅ 所有指定狀態和其他未列舉狀態的記錄已成功刪除")
    else:
        print(f"⚠️ 仍有 {remaining_delete_statuses} 筆應刪除的記錄未刪除")
else:
    print("⚠️ 沒有找到需要刪除的狀態記錄")

# 檢查處理後的結果
print(f"\n=== 處理後結果 ===")
print(f"處理後d資料集筆數: {len(d)}")
print(f"筆數變化: {len(d) - len(status_before)} (刪除了 {len(status_before) - len(d)} 筆)")

status_after = d['使用狀態'].value_counts()
print(f"\n處理後使用狀態分布:")
print(status_after)

# 對比處理前後的變化
print(f"\n=== 處理前後對比 ===")
all_statuses = set(status_before.index) | set(status_after.index)

for status in sorted(all_statuses):
    before_count = status_before.get(status, 0)
    after_count = status_after.get(status, 0)
    change = after_count - before_count
    
    if change != 0:
        change_str = f"({change:+d})" if change != 0 else ""
        print(f"{status}: {before_count} → {after_count} {change_str}")

# 重設索引
d = d.reset_index(drop=True)
print(f"\n✅ 已重設索引，最終d資料集形狀: {d.shape}")

# 驗證最終狀態
print(f"\n=== 最終驗證 ===")
final_statuses = d['使用狀態'].unique()
print(f"最終包含的使用狀態: {sorted(final_statuses)}")

# 檢查最終狀態是否只包含預期的狀態
expected_final_statuses = {'使用中', '停用'}
unexpected_statuses = set(final_statuses) - expected_final_statuses

if unexpected_statuses:
    print(f"⚠️ 警告：存在非預期的狀態: {unexpected_statuses}")
else:
    print("✅ 最終狀態符合預期，只包含'使用中'和'停用'")

# 顯示最終統計
print(f"\n=== 最終統計 ===")
print(f"使用中客戶數: {(d['使用狀態'] == '使用中').sum()}")
print(f"停用客戶數: {(d['使用狀態'] == '停用').sum()}")
print(f"使用中比例: {(d['使用狀態'] == '使用中').mean() * 100:.1f}%")
print(f"停用比例: {(d['使用狀態'] == '停用').mean() * 100:.1f}%")

=== 針對d資料集處理使用狀態 ===
=== 處理前使用狀態分布 ===
處理前d資料集筆數: 321993
處理前使用狀態分布:
使用狀態
停用            183326
使用中           107127
無法完工           16165
欠款斷線           12563
拆機中             2087
維修中              348
設備加裝             254
派收中               44
暫停                36
移機中               31
裝機中                7
促銷中                4
退租(先斷線未拆機)         1
Name: count, dtype: int64

=== 狀態處理規則 ===
1. 改為停用: 欠款斷線、拆機中、退租(先斷線未拆機)
2. 改為使用中: 維修中、設備加裝、派收中、暫停、移機中
3. 刪除記錄: 無法完工、裝機中、促銷中，其他未列舉狀態

=== 狀態分析 ===
將改為停用的狀態: ['欠款斷線', '拆機中', '退租(先斷線未拆機)']
將改為使用中的狀態: ['維修中', '設備加裝', '派收中', '暫停', '移機中']
明確要刪除的狀態: ['無法完工', '裝機中', '促銷中']
保持不變的狀態: ['使用中', '停用']

=== 發現的其他未列舉狀態 ===
  ✅ 沒有發現其他未列舉的狀態

=== 步驟1: 將指定狀態改為停用 ===
  欠款斷線: 12563 筆
  拆機中: 2087 筆
  退租(先斷線未拆機): 1 筆
✅ 已將 3 種狀態改為停用

=== 步驟2: 將指定狀態改為使用中 ===
  維修中: 348 筆
  設備加裝: 254 筆
  派收中: 44 筆
  暫停: 36 筆
  移機中: 31 筆
✅ 已將 5 種狀態改為使用中

=== 步驟3: 刪除指定狀態和其他未列舉狀態的記錄 ===
要刪除的狀態:
  無法完工: 16165 筆
  裝機中: 7 筆
  促銷中: 4 筆
總計需刪除: 16176 筆
✅ 已刪除 16176 筆記錄
✅ 所有指定狀態和其他未列舉狀態的記錄已成功刪除

=== 

In [9]:
# 在d資料集中建立alive_month欄位
print("=== 在d資料集中建立alive_month欄位 ===")

# 檢查當前d資料集狀況
print(f"d資料集形狀: {d.shape}")
print(f"起日欄位資料型態: {d['起日'].dtype}")
print(f"迄日欄位資料型態: {d['迄日'].dtype}")

# 檢查起日和迄日的範例值
print(f"\n=== 起日和迄日範例值 ===")
print("起日範例值:")
print(d['起日'].head(10).tolist())
print("\n迄日範例值 (含空值):")
print(d['迄日'].head(10).tolist())
print(f"\n迄日空值數量: {d['迄日'].isnull().sum()}")
print(f"迄日空值比例: {d['迄日'].isnull().mean() * 100:.1f}%")

# 步驟1: 轉換起日為日期格式
print(f"\n=== 步驟1: 轉換起日為日期格式 ===")
if d['起日'].dtype == 'object':
    try:
        # 嘗試直接轉換
        d['起日_datetime'] = pd.to_datetime(d['起日'], errors='coerce')
    except:
        # 如果失敗，嘗試指定格式
        d['起日_datetime'] = pd.to_datetime(d['起日'], format='%Y%m%d', errors='coerce')
    
    # 檢查轉換結果
    null_start_dates = d['起日_datetime'].isnull().sum()
    if null_start_dates > 0:
        print(f"⚠️ 有 {null_start_dates} 筆起日無法轉換為日期格式")
        # 顯示無法轉換的起日範例
        invalid_dates = d[d['起日_datetime'].isnull()]['起日'].unique()[:5]
        print(f"無法轉換的起日範例: {list(invalid_dates)}")
    else:
        print("✅ 所有起日都成功轉換為日期格式")
else:
    d['起日_datetime'] = pd.to_datetime(d['起日'], errors='coerce')

# 步驟2: 處理迄日
print(f"\n=== 步驟2: 處理迄日 ===")
print("處理邏輯: 迄日為空值時，使用2024/12/31作為結束日期")

# 設定基準日期
base_date = pd.to_datetime('2024-12-31')
print(f"基準日期: {base_date.strftime('%Y-%m-%d')}")

# 處理迄日：空值用基準日期填補，非空值轉為日期格式
def process_end_date(end_date_value):
    if pd.isnull(end_date_value):
        return base_date
    else:
        try:
            return pd.to_datetime(end_date_value, errors='coerce')
        except:
            try:
                return pd.to_datetime(str(end_date_value), format='%Y%m%d', errors='coerce')
            except:
                return base_date  # 如果轉換失敗，使用基準日期

d['迄日_datetime'] = d['迄日'].apply(process_end_date)

# 檢查迄日處理結果
print(f"迄日轉換完成，空值處理情況:")
print(f"原始迄日空值數: {d['迄日'].isnull().sum()}")
print(f"處理後迄日_datetime空值數: {d['迄日_datetime'].isnull().sum()}")

# 步驟3: 計算存活月數
print(f"\n=== 步驟3: 計算存活月數 ===")

def calculate_alive_months(start_date, end_date):
    """
    計算兩個日期之間的月數差異
    """
    if pd.isnull(start_date) or pd.isnull(end_date):
        return 0
    
    # 計算年份和月份的差異
    year_diff = end_date.year - start_date.year
    month_diff = end_date.month - start_date.month
    day_diff = end_date.day - start_date.day
    
    # 總月數差異
    total_months = year_diff * 12 + month_diff
    
    # 如果結束日期的日數小於開始日期的日數，減去1個月
    if day_diff < 0:
        total_months -= 1
    
    # 確保最小值為0
    return max(0, total_months)

# 計算存活月數
d['alive_month'] = d.apply(
    lambda row: calculate_alive_months(row['起日_datetime'], row['迄日_datetime']), 
    axis=1
)

print(f"✅ 存活月數計算完成")

# 步驟4: 驗證和分析結果
print(f"\n=== 步驟4: 驗證和分析結果 ===")

# 基本統計
print(f"存活月數統計:")
print(f"  最小值: {d['alive_month'].min()}")
print(f"  最大值: {d['alive_month'].max()}")
print(f"  平均值: {d['alive_month'].mean():.2f}")
print(f"  中位數: {d['alive_month'].median():.2f}")
print(f"  標準差: {d['alive_month'].std():.2f}")

# 存活月數分布
print(f"\n存活月數分布 (前20個值):")
alive_distribution = d['alive_month'].value_counts().sort_index()
print(alive_distribution.head(20))

# 檢查特殊情況
print(f"\n=== 特殊情況檢查 ===")

# 存活月數為0的情況
zero_months = (d['alive_month'] == 0).sum()
print(f"存活月數為0的記錄數: {zero_months}")

if zero_months > 0:
    print("存活月數為0的範例 (前5筆):")
    zero_examples = d[d['alive_month'] == 0][['CUST_NO', '起日', '迄日', '起日_datetime', '迄日_datetime', 'alive_month']].head()
    for idx, row in zero_examples.iterrows():
        print(f"  客編: {row['CUST_NO']}, 起日: {row['起日']}, 迄日: {row['迄日']}, 存活月數: {row['alive_month']}")

# 使用基準日期的情況（原本迄日為空值）
used_base_date = (d['迄日'].isnull()).sum()
print(f"\n使用基準日期(2024/12/31)計算的記錄數: {used_base_date}")

if used_base_date > 0:
    print("使用基準日期的範例 (前5筆):")
    base_date_examples = d[d['迄日'].isnull()][['CUST_NO', '起日', '迄日', 'alive_month']].head()
    for idx, row in base_date_examples.iterrows():
        print(f"  客編: {row['CUST_NO']}, 起日: {row['起日']}, 迄日: {row['迄日']}, 存活月數: {row['alive_month']}")

# 長期用戶分析
long_term_threshold = 60  # 5年以上
long_term_users = (d['alive_month'] >= long_term_threshold).sum()
print(f"\n長期用戶 (存活{long_term_threshold}個月以上): {long_term_users} 位")
print(f"長期用戶比例: {long_term_users / len(d) * 100:.1f}%")

# 按存活月數區間統計
print(f"\n=== 存活月數區間統計 ===")
def categorize_alive_months(months):
    if months == 0:
        return '0個月'
    elif months <= 6:
        return '1-6個月'
    elif months <= 12:
        return '7-12個月'
    elif months <= 24:
        return '1-2年'
    elif months <= 36:
        return '2-3年'
    elif months <= 60:
        return '3-5年'
    else:
        return '5年以上'

d['alive_month_category'] = d['alive_month'].apply(categorize_alive_months)
category_distribution = d['alive_month_category'].value_counts()

print("存活月數區間分布:")
for category, count in category_distribution.items():
    percentage = count / len(d) * 100
    print(f"  {category}: {count:,} 位 ({percentage:.1f}%)")

# 清理臨時欄位（保留alive_month）
columns_to_drop = ['起日_datetime', '迄日_datetime', 'alive_month_category']
existing_temp_cols = [col for col in columns_to_drop if col in d.columns]

if existing_temp_cols:
    d = d.drop(columns=existing_temp_cols)
    print(f"\n✅ 已清理臨時欄位: {existing_temp_cols}")

print(f"\n✅ alive_month欄位建立完成")
print(f"最終d資料集形狀: {d.shape}")
print(f"新增欄位: alive_month")

# 顯示最終結果範例
print(f"\n=== 最終結果範例 ===")
sample_data = d[['CUST_NO', '起日', '迄日', 'alive_month', '使用狀態']].head(10)
print(sample_data.to_string(index=False))

=== 在d資料集中建立alive_month欄位 ===
d資料集形狀: (305817, 10)
起日欄位資料型態: object
迄日欄位資料型態: object

=== 起日和迄日範例值 ===
起日範例值:
['2021/10/29 19:10:30', '2023/09/13 11:53:01', '2022/06/08 11:10:05', '2019/06/21 10:56:15', '2023/11/06 13:53:11', '2021/07/10 20:27:26', '2021/10/04 11:56:24', '2016/07/16 21:56:13', '2019/11/22 17:07:55', '2021/03/23 14:29:58']

迄日範例值 (含空值):
[nan, nan, nan, nan, nan, nan, nan, '2017/04/06 11:49:08', nan, nan]

迄日空值數量: 127020
迄日空值比例: 41.5%

=== 步驟1: 轉換起日為日期格式 ===
✅ 所有起日都成功轉換為日期格式

=== 步驟2: 處理迄日 ===
處理邏輯: 迄日為空值時，使用2024/12/31作為結束日期
基準日期: 2024-12-31
迄日轉換完成，空值處理情況:
原始迄日空值數: 127020
處理後迄日_datetime空值數: 0

=== 步驟3: 計算存活月數 ===
✅ 存活月數計算完成

=== 步驟4: 驗證和分析結果 ===
存活月數統計:
  最小值: 0
  最大值: 230
  平均值: 43.55
  中位數: 31.00
  標準差: 42.05

存活月數分布 (前20個值):
alive_month
0     28066
1      2325
2      2941
3      2943
4      2850
5      4214
6      8728
7      5788
8      3759
9      4086
10     3280
11     5314
12    11282
13     6674
14     4211
15     3969
16     3313
17     3539
18     4314
19     

In [11]:
# 儲存整理過後的d資料集
print("=== 儲存整理過後的d資料集 ===")

# 檢查當前d資料集狀況
print(f"當前d資料集形狀: {d.shape}")
print(f"當前d資料集欄位數: {len(d.columns)}")

# 顯示d資料集的欄位列表
print(f"\n=== d資料集欄位列表 ===")
for i, col in enumerate(d.columns, 1):
    print(f"{i:2d}. {col}")

# 檢查關鍵統計資訊
print(f"\n=== d資料集關鍵統計 ===")
print(f"客戶數量: {d['CUST_NO'].nunique()}")
print(f"總記錄數: {len(d)}")
print(f"產品名稱分布:")
print(d['產品名稱'].value_counts())
print(f"\n使用狀態分布:")
print(d['使用狀態'].value_counts())

# 檢查alive_month欄位統計
if 'alive_month' in d.columns:
    print(f"\nalive_month統計:")
    print(f"  最小值: {d['alive_month'].min()}")
    print(f"  最大值: {d['alive_month'].max()}")
    print(f"  平均值: {d['alive_month'].mean():.2f}")
    print(f"  中位數: {d['alive_month'].median():.2f}")

# 檢查空值情況
null_count = d.isnull().sum().sum()
print(f"\n總空值數: {null_count}")

# 設定輸出檔案路徑
output_path = 'cleaned_dataset3.csv'

# 儲存資料集
try:
    d.to_csv(output_path, sep='^', index=False)
    print(f"\n✅ 已成功儲存d資料集至 {output_path}")
    print(f"儲存格式: CSV檔案，分隔符號為 '^'")
    print(f"是否包含索引: 否")
    
    # 驗證儲存結果
    import os
    if os.path.exists(output_path):
        file_size = os.path.getsize(output_path)
        print(f"檔案大小: {file_size:,} bytes ({file_size/1024/1024:.2f} MB)")
    
except Exception as e:
    print(f"❌ 儲存失敗: {str(e)}")

# 顯示儲存的資料摘要
print(f"\n=== 儲存的資料摘要 ===")
print(f"檔案名稱: {output_path}")
print(f"資料筆數: {len(d):,}")
print(f"欄位數量: {len(d.columns)}")
print(f"唯一客戶數: {d['CUST_NO'].nunique():,}")
print(f"產品類型: {', '.join(d['產品名稱'].unique())}")
print(f"使用狀態: {', '.join(d['使用狀態'].unique())}")

# 顯示前5筆資料作為範例
print(f"\n=== 儲存資料範例（前5筆） ===")
sample_cols = ['CUST_NO', '產品名稱', '使用狀態', '起日', '迄日']
if 'alive_month' in d.columns:
    sample_cols.append('alive_month')

print(d[sample_cols].head().to_string(index=False))

=== 儲存整理過後的d資料集 ===
當前d資料集形狀: (305817, 11)
當前d資料集欄位數: 11

=== d資料集欄位列表 ===
 1. CUST_NO
 2. 產品名稱
 3. 用戶種類
 4. 相關編號
 5. 起日
 6. 迄日
 7. 系統台
 8. 地區
 9. 繳別
10. 使用狀態
11. alive_month

=== d資料集關鍵統計 ===
客戶數量: 305817
總記錄數: 305817
產品名稱分布:
產品名稱
CM      200482
EPON    105335
Name: count, dtype: int64

使用狀態分布:
使用狀態
停用     197977
使用中    107840
Name: count, dtype: int64

alive_month統計:
  最小值: 0
  最大值: 230
  平均值: 43.55
  中位數: 31.00

總空值數: 127029

✅ 已成功儲存d資料集至 cleaned_dataset3.csv
儲存格式: CSV檔案，分隔符號為 '^'
是否包含索引: 否
檔案大小: 30,363,879 bytes (28.96 MB)

=== 儲存的資料摘要 ===
檔案名稱: cleaned_dataset3.csv
資料筆數: 305,817
欄位數量: 11
唯一客戶數: 305,817
產品類型: EPON, CM
使用狀態: 使用中, 停用

=== 儲存資料範例（前5筆） ===
 CUST_NO 產品名稱 使用狀態                  起日  迄日  alive_month
       4 EPON  使用中 2021/10/29 19:10:30 NaN           38
      35 EPON  使用中 2023/09/13 11:53:01 NaN           15
      47 EPON  使用中 2022/06/08 11:10:05 NaN           30
      96 EPON   停用 2019/06/21 10:56:15 NaN           66
     110 EPON  使用中 2023/11/06 13:53:11 NaN           13

In [ ]:

# --- B 類別重組 ---
b['MAIN_CATEGORY'] = b['MAIN_CATEGORY'].where(
    b['MAIN_CATEGORY'].isin(b['MAIN_CATEGORY'].value_counts().nlargest(4).index),
    '其他主類')
b['SUB_CATEGORY'] = b['SUB_CATEGORY'].where(
    b['SUB_CATEGORY'].isin(b['SUB_CATEGORY'].value_counts().nlargest(6).index),
    '其他分類')
b_encoded = pd.get_dummies(b[['MAIN_CATEGORY','SUB_CATEGORY']],
                           prefix=['maincat','subcat'], dtype=np.int8)
b = pd.concat([b.drop(columns=['MAIN_CATEGORY','SUB_CATEGORY']), b_encoded], axis=1)

# --- C 使用狀態對照表 (EPON/CM) ---
cols = ['客編','產品名稱','用戶種類','相關編號','起日','迄日',
        '系統台','地區','繳別','使用狀態']
status_map = {}
for chunk in pd.read_csv(c_path, sep='^', engine='python',
                         names=cols, dtype=str, chunksize=300_000):
    sub = chunk[chunk['產品名稱'].isin(['EPON','CM'])]
    sub['status_val'] = np.where(sub['使用狀態']=='使用中', 0, 1)
    status_map.update(sub.groupby('客編')['status_val'].min().to_dict())

# --- 合併 A+B ---
merged = b.merge(a, on='CUST_NO', how='outer')

# --- 補齊使用狀態_數值 ---
mask = merged['使用狀態_數值'].isna()
merged.loc[mask, '使用狀態_數值'] = merged.loc[mask, 'CUST_NO'].map(status_map)

# --- 其餘空值補 0 ---
merged = merged.fillna(0)

# --- 輸出 ---
merged.to_csv(out_path, index=False)
print(f'Done → {out_path} , shape={merged.shape}')


Enhance process start here

In [9]:
import pandas as pd, numpy as np

# === 路徑 ===
a_path = 'ds_doService_combo_filtered_top10.csv'   # A 工單特徵
b_path = 'simplified_customer_features_top10.csv'  # B 來電記錄特徵
c_path = 'ds_billing_onehot.csv'                    # C 帳單資料特徵


# === A, B ===
a = pd.read_csv(a_path,sep='^').rename(columns={'客編': 'CUST_NO'})
b = pd.read_csv(b_path,sep=',')
c = pd.read_csv(c_path,sep=',').rename(columns={'客編': 'CUST_NO'})

In [10]:
a.head()

,CUST_NO,工單嚴重程度_平均,工單嚴重程度_最高,工單升級趨勢,產品名稱,使用狀態,最新工單日期,工單日期,30天內,60天內,...,工單原因_Epon實體問題_光纖斷裂_是否發生,工單原因_RF線路問題_調整訊號_次數,工單原因_RF線路問題_調整訊號_佔比,工單原因_RF線路問題_調整訊號_是否發生,工單原因_Epon實體問題_光纖斷裂_佔比,工單原因_Epon實體問題_光纖斷裂_次數,工單原因_其他_次數,工單原因_其他_是否發生,"工單原因_用戶端問題_數據機故,更換_次數","工單原因_用戶端問題_數據機故,更換_是否發生"
0,4,2.0,2,0.0,EPON,使用中,2023-09-04 21:19:20,2023-09-04 21:19:20,1,0,...,0,0,0.0,0,0.0,0,0,0,0,0
1,47,2.0,2,0.0,EPON,使用中,2022-08-26 20:58:06,2022-08-26 20:58:06,1,0,...,0,0,0.0,0,0.0,0,0,0,0,0
2,138,2.0,2,0.0,EPON,使用中,2022-04-13 21:29:02,2022-04-13 21:29:02,1,0,...,0,0,0.0,0,0.0,0,0,0,0,0
3,415,2.0,2,0.0,CM,使用中,2024-01-24 22:44:05,2024-01-24 22:44:05,1,0,...,0,0,0.0,0,0.0,0,0,0,0,0
4,503,2.0,2,0.0,EPON,使用中,2024-05-03 20:05:51,2024-05-03 20:05:51,1,0,...,0,0,0.0,0,0.0,0,0,0,0,0


In [11]:
a.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 49871 entries, 0 to 49870
Data columns (total 23 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   CUST_NO                  49871 non-null  int64  
 1   工單嚴重程度_平均                49871 non-null  float64
 2   工單嚴重程度_最高                49871 non-null  int64  
 3   工單升級趨勢                   49871 non-null  float64
 4   產品名稱                     49871 non-null  object 
 5   使用狀態                     49871 non-null  object 
 6   最新工單日期                   49871 non-null  object 
 7   工單日期                     49871 non-null  object 
 8   30天內                     49871 non-null  int64  
 9   60天內                     49871 non-null  int64  
 10  90天內                     49871 non-null  int64  
 11  90天以上                    49871 non-null  int64  
 12  平均等待天數                   49871 non-null  float64
 13  工單原因_Epon實體問題_光纖斷裂_是否發生  49871 non-null  int64  
 14  工單原因_RF線路問題_調整訊號_次數   

In [12]:
a['產品名稱'].value_counts()

產品名稱
EPON    26614
CM      23257
Name: count, dtype: int64

In [13]:
# 篩選a資料集中僅保留產品名稱為EPON或CM的資料
print("=== 篩選前a資料集狀況 ===")
print(f"篩選前a資料集筆數: {len(a)}")
print("產品名稱分布:")
print(a['產品名稱'].value_counts())

# 篩選僅保留EPON或CM的資料
a_filtered = a[a['產品名稱'].isin(['EPON', 'CM'])].copy()

print(f"\n=== 篩選後a資料集狀況 ===")
print(f"篩選後a資料集筆數: {len(a_filtered)}")
print(f"刪除筆數: {len(a) - len(a_filtered)}")
print(f"保留比例: {len(a_filtered) / len(a) * 100:.1f}%")
print("篩選後產品名稱分布:")
print(a_filtered['產品名稱'].value_counts())

# 更新a變數
a = a_filtered.copy()

print(f"\n✅ 已成功篩選a資料集，僅保留產品名稱為EPON或CM的資料")
print(f"更新後a資料集形狀: {a.shape}")

=== 篩選前a資料集狀況 ===
篩選前a資料集筆數: 49871
產品名稱分布:
產品名稱
EPON    26614
CM      23257
Name: count, dtype: int64

=== 篩選後a資料集狀況 ===
篩選後a資料集筆數: 49871
刪除筆數: 0
保留比例: 100.0%
篩選後產品名稱分布:
產品名稱
EPON    26614
CM      23257
Name: count, dtype: int64

✅ 已成功篩選a資料集，僅保留產品名稱為EPON或CM的資料
更新後a資料集形狀: (49871, 23)


In [14]:
# 對產品名稱進行one-hot編碼
print("=== 對產品名稱進行one-hot編碼 ===")

# 顯示編碼前的狀況
print("編碼前產品名稱分布:")
print(a['產品名稱'].value_counts())
print(f"編碼前a資料集形狀: {a.shape}")

# 進行one-hot編碼
product_encoded = pd.get_dummies(a['產品名稱'], prefix='product', dtype=int)

# 將編碼後的欄位合併到原資料集，並移除原始的'產品名稱'欄位
a_encoded = pd.concat([a.drop(columns=['產品名稱']), product_encoded], axis=1)

print(f"\n=== one-hot編碼結果 ===")
print(f"新增的one-hot欄位: {list(product_encoded.columns)}")
print(f"編碼後a資料集形狀: {a_encoded.shape}")
print(f"新增欄位數: {len(product_encoded.columns)}")

# 顯示新增欄位的分布
print(f"\n各one-hot欄位的值分布:")
for col in product_encoded.columns:
    print(f"{col}: {a_encoded[col].sum()} 筆 (比例: {a_encoded[col].mean()*100:.1f}%)")

# 更新a變數
a = a_encoded.copy()

print(f"\n✅ 已成功對產品名稱進行one-hot編碼")
print(f"更新後a資料集形狀: {a.shape}")

=== 對產品名稱進行one-hot編碼 ===
編碼前產品名稱分布:
產品名稱
EPON    26614
CM      23257
Name: count, dtype: int64
編碼前a資料集形狀: (49871, 23)

=== one-hot編碼結果 ===
新增的one-hot欄位: ['product_CM', 'product_EPON']
編碼後a資料集形狀: (49871, 24)
新增欄位數: 2

各one-hot欄位的值分布:
product_CM: 23257 筆 (比例: 46.6%)
product_EPON: 26614 筆 (比例: 53.4%)

✅ 已成功對產品名稱進行one-hot編碼
更新後a資料集形狀: (49871, 24)


In [17]:
a.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 49871 entries, 0 to 49870
Data columns (total 24 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   CUST_NO                  49871 non-null  int64  
 1   工單嚴重程度_平均                49871 non-null  float64
 2   工單嚴重程度_最高                49871 non-null  int64  
 3   工單升級趨勢                   49871 non-null  float64
 4   使用狀態                     49871 non-null  object 
 5   最新工單日期                   49871 non-null  object 
 6   工單日期                     49871 non-null  object 
 7   30天內                     49871 non-null  int64  
 8   60天內                     49871 non-null  int64  
 9   90天內                     49871 non-null  int64  
 10  90天以上                    49871 non-null  int64  
 11  平均等待天數                   49871 non-null  float64
 12  工單原因_Epon實體問題_光纖斷裂_是否發生  49871 non-null  int64  
 13  工單原因_RF線路問題_調整訊號_次數      49871 non-null  int64  
 14  工單原因_RF線路問題_調整訊號_佔比   

In [18]:
# 刪除a資料集中的指定欄位
print("=== 刪除a資料集中的指定欄位 ===")

# 要刪除的欄位列表
columns_to_drop = ['工單日期', '最新工單日期']

# 檢查刪除前的狀況
print(f"刪除前a資料集形狀: {a.shape}")
print(f"刪除前總欄位數: {len(a.columns)}")

# 檢查哪些欄位實際存在於資料集中
existing_columns = [col for col in columns_to_drop if col in a.columns]
missing_columns = [col for col in columns_to_drop if col not in a.columns]

print(f"\n=== 欄位檢查結果 ===")
print(f"存在的欄位: {existing_columns}")
if missing_columns:
    print(f"不存在的欄位: {missing_columns}")

# 執行刪除操作（僅刪除存在的欄位）
if existing_columns:
    a_cleaned = a.drop(columns=existing_columns)
    print(f"\n=== 刪除結果 ===")
    print(f"實際刪除的欄位: {existing_columns}")
    print(f"刪除的欄位數: {len(existing_columns)}")
    print(f"刪除後a資料集形狀: {a_cleaned.shape}")
    print(f"刪除後總欄位數: {len(a_cleaned.columns)}")
    
    # 更新a變數
    a = a_cleaned.copy()
    
    print(f"\n✅ 已成功刪除指定欄位")
else:
    print(f"\n⚠️ 沒有找到要刪除的欄位")

=== 刪除a資料集中的指定欄位 ===
刪除前a資料集形狀: (49871, 24)
刪除前總欄位數: 24

=== 欄位檢查結果 ===
存在的欄位: ['工單日期', '最新工單日期']

=== 刪除結果 ===
實際刪除的欄位: ['工單日期', '最新工單日期']
刪除的欄位數: 2
刪除後a資料集形狀: (49871, 22)
刪除後總欄位數: 22

✅ 已成功刪除指定欄位


In [19]:
# 顯示刪除後的資料集資訊
print("=== 刪除後資料集資訊 ===")
print(f"當前a資料集形狀: {a.shape}")
print(f"剩餘欄位:")
for i, col in enumerate(a.columns, 1):
    print(f"{i:2d}. {col}")

=== 刪除後資料集資訊 ===
當前a資料集形狀: (49871, 22)
剩餘欄位:
 1. CUST_NO
 2. 工單嚴重程度_平均
 3. 工單嚴重程度_最高
 4. 工單升級趨勢
 5. 使用狀態
 6. 30天內
 7. 60天內
 8. 90天內
 9. 90天以上
10. 平均等待天數
11. 工單原因_Epon實體問題_光纖斷裂_是否發生
12. 工單原因_RF線路問題_調整訊號_次數
13. 工單原因_RF線路問題_調整訊號_佔比
14. 工單原因_RF線路問題_調整訊號_是否發生
15. 工單原因_Epon實體問題_光纖斷裂_佔比
16. 工單原因_Epon實體問題_光纖斷裂_次數
17. 工單原因_其他_次數
18. 工單原因_其他_是否發生
19. 工單原因_用戶端問題_數據機故,更換_次數
20. 工單原因_用戶端問題_數據機故,更換_是否發生
21. product_CM
22. product_EPON


In [20]:
b.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 44349 entries, 0 to 44348
Data columns (total 53 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   MAX_SENTIMENT_SCORE        44349 non-null  int64  
 1   PROBLEM_COMPLEXITY_SCORE   44349 non-null  float64
 2   SUB_CAT_方案異動CM轉EOC_COUNT   44349 non-null  float64
 3   AVG_CALL_INTERVAL_DAYS     44349 non-null  float64
 4   MAIN_CAT_非連線問題_COUNT       44349 non-null  float64
 5   SUB_CATEGORY_DIVERSITY     44349 non-null  int64  
 6   MAIN_CAT_新產品_COUNT         44349 non-null  float64
 7   SUB_CAT_建議與抱怨_RATIO        44349 non-null  float64
 8   SUB_CAT_方案異動CM轉EOC_RATIO   44349 non-null  float64
 9   TOTAL_CALLS                44349 non-null  int64  
 10  STD_CALL_INTERVAL_DAYS     44349 non-null  float64
 11  ESCALATION_RISK            44349 non-null  float64
 12  SUB_CAT_建議與抱怨_COUNT        44349 non-null  float64
 13  TOTAL_SENTIMENT_SCORE      44349 non-null  int

In [21]:
c.head()

,CUST_NO,平均繳款延遲日數,paytype_1,paytype_3,paytype_6,paytype_12,paytype_15,paytype_99,paymethod_APP臨櫃代收(7-11),paymethod_CVS臨櫃代收(7-11),paymethod_信用卡扣款,paymethod_其他繳款方式,paymethod_廠商代收,paymethod_當月繳帳單,paymethod_簡訊帳單臨櫃代收(7-11),paymethod_金融機構轉帳
0,895701,-46.700000,0,0,0,1,0,0,0,0,0,1,0,0,0,0
1,1129002,19.333333,1,0,0,0,0,0,0,0,0,1,0,0,0,0
2,223725,13.420000,0,0,0,1,0,0,0,1,0,0,0,0,0,0
3,533302,-43.285714,0,1,0,0,0,0,0,0,0,0,0,0,1,0
4,1026745,-34.800000,0,0,0,1,0,0,0,0,0,0,0,0,1,0


In [22]:
# 以b資料集為基礎進行資料合併
print("=== 以b資料集為基礎進行資料合併 ===")

# 顯示合併前各資料集的狀況
print("=== 合併前各資料集狀況 ===")
print(f"b資料集 (基礎): {b.shape}")
print(f"a資料集 (工單特徵): {a.shape}")
print(f"c資料集 (帳單特徵): {c.shape}")

print(f"\nb資料集CUST_NO數量: {b['CUST_NO'].nunique()}")
print(f"a資料集CUST_NO數量: {a['CUST_NO'].nunique()}")
print(f"c資料集CUST_NO數量: {c['CUST_NO'].nunique()}")

# 檢查重疊的CUST_NO
b_cust_no = set(b['CUST_NO'])
a_cust_no = set(a['CUST_NO'])
c_cust_no = set(c['CUST_NO'])

a_overlap = a_cust_no & b_cust_no
c_overlap = c_cust_no & b_cust_no

print(f"\n=== CUST_NO重疊分析 ===")
print(f"a與b重疊的CUST_NO數量: {len(a_overlap)}")
print(f"c與b重疊的CUST_NO數量: {len(c_overlap)}")
print(f"a與b重疊比例: {len(a_overlap) / len(b_cust_no) * 100:.1f}%")
print(f"c與b重疊比例: {len(c_overlap) / len(b_cust_no) * 100:.1f}%")

=== 以b資料集為基礎進行資料合併 ===
=== 合併前各資料集狀況 ===
b資料集 (基礎): (44349, 53)
a資料集 (工單特徵): (49871, 22)
c資料集 (帳單特徵): (346407, 16)

b資料集CUST_NO數量: 44349
a資料集CUST_NO數量: 49871
c資料集CUST_NO數量: 346407

=== CUST_NO重疊分析 ===
a與b重疊的CUST_NO數量: 25503
c與b重疊的CUST_NO數量: 38046
a與b重疊比例: 57.5%
c與b重疊比例: 85.8%


In [40]:
# 步驟1: 將a資料集中存在於b資料集的CUST_NO合併至b
print("\n=== 步驟1: 合併a資料集 ===")

# 篩選a資料集中存在於b資料集的客戶
a_to_merge = a[a['CUST_NO'].isin(b['CUST_NO'])].copy()
print(f"從a資料集中篩選出可合併的筆數: {len(a_to_merge)}")

# 執行左連接 (以b為主，保留b的所有記錄)
merged_step1 = b.merge(a_to_merge, on='CUST_NO', how='left')
print(f"合併a後的資料集形狀: {merged_step1.shape}")

# 檢查合併後的空值情況
a_columns = [col for col in a_to_merge.columns if col != 'CUST_NO']
print(f"來自a資料集的新欄位數: {len(a_columns)}")
print(f"新欄位中的空值統計:")
for col in a_columns[:5]:  # 顯示前5個欄位的空值情況
    null_count = merged_step1[col].isnull().sum()
    print(f"  {col}: {null_count} 個空值 ({null_count/len(merged_step1)*100:.1f}%)")
if len(a_columns) > 5:
    print(f"  ... 還有 {len(a_columns)-5} 個欄位")


=== 步驟1: 合併a資料集 ===
從a資料集中篩選出可合併的筆數: 25503
合併a後的資料集形狀: (44349, 74)
來自a資料集的新欄位數: 21
新欄位中的空值統計:
  工單嚴重程度_平均: 18846 個空值 (42.5%)
  工單嚴重程度_最高: 18846 個空值 (42.5%)
  工單升級趨勢: 18846 個空值 (42.5%)
  使用狀態: 18846 個空值 (42.5%)
  30天內: 18846 個空值 (42.5%)
  ... 還有 16 個欄位


In [41]:
# 步驟2: 將c資料集中存在於b資料集的CUST_NO合併至已合併的資料
print("\n=== 步驟2: 合併c資料集 ===")

# 篩選c資料集中存在於b資料集的客戶
c_to_merge = c[c['CUST_NO'].isin(b['CUST_NO'])].copy()
print(f"從c資料集中篩選出可合併的筆數: {len(c_to_merge)}")

# 執行左連接 (以step1結果為主)
final_merged = merged_step1.merge(c_to_merge, on='CUST_NO', how='left')
print(f"最終合併後的資料集形狀: {final_merged.shape}")

# 檢查合併後的空值情況
c_columns = [col for col in c_to_merge.columns if col != 'CUST_NO']
print(f"來自c資料集的新欄位數: {len(c_columns)}")
print(f"新欄位中的空值統計:")
for col in c_columns[:5]:  # 顯示前5個欄位的空值情況
    null_count = final_merged[col].isnull().sum()
    print(f"  {col}: {null_count} 個空值 ({null_count/len(final_merged)*100:.1f}%)")
if len(c_columns) > 5:
    print(f"  ... 還有 {len(c_columns)-5} 個欄位")


=== 步驟2: 合併c資料集 ===
從c資料集中篩選出可合併的筆數: 38046
最終合併後的資料集形狀: (44349, 89)
來自c資料集的新欄位數: 15
新欄位中的空值統計:
  平均繳款延遲日數: 6303 個空值 (14.2%)
  paytype_1: 6303 個空值 (14.2%)
  paytype_3: 6303 個空值 (14.2%)
  paytype_6: 6303 個空值 (14.2%)
  paytype_12: 6303 個空值 (14.2%)
  ... 還有 10 個欄位


In [43]:
# 檢查最終合併結果
print("\n=== 最終合併結果摘要 ===")
print(f"原始b資料集: {b.shape}")
print(f"最終合併資料集: {final_merged.shape}")
print(f"新增欄位數: {final_merged.shape[1] - b.shape[1]}")
print(f"客戶數量保持不變: {len(final_merged) == len(b)}")

# 檢查總體空值情況
total_nulls = final_merged.isnull().sum().sum()
total_cells = final_merged.shape[0] * final_merged.shape[1]
print(f"總空值數: {total_nulls:,}")
print(f"總空值比例: {total_nulls/total_cells*100:.2f}%")

# 顯示欄位分組
original_b_cols = list(b.columns)
a_cols = [col for col in final_merged.columns if col in a.columns and col != 'CUST_NO']
c_cols = [col for col in final_merged.columns if col in c.columns and col != 'CUST_NO']

print(f"\n=== 欄位分組 ===")
print(f"原始b資料集欄位: {len(original_b_cols)} 個")
print(f"來自a資料集欄位: {len(a_cols)} 個")
print(f"來自c資料集欄位: {len(c_cols)} 個")
print(f"總欄位數: {final_merged.shape[1]} 個")


=== 最終合併結果摘要 ===
原始b資料集: (44349, 53)
最終合併資料集: (44349, 89)
新增欄位數: 36
客戶數量保持不變: True
總空值數: 490,311
總空值比例: 12.42%

=== 欄位分組 ===
原始b資料集欄位: 53 個
來自a資料集欄位: 21 個
來自c資料集欄位: 15 個
總欄位數: 89 個


In [42]:
final_merged['使用狀態'].value_counts()

使用狀態
使用中     22853
停用       1640
欠款斷線      563
維修中       203
拆機中       156
設備加裝       66
暫停         11
移機中         8
派收中         3
Name: count, dtype: int64

改變合併的方向，從原本的「將a合併至b」改為「將b合併至a」

In [44]:
# 步驟1: 將b資料集中存在於a資料集的CUST_NO合併至a
print("\n=== 步驟1: 合併b資料集至a ===")

# 篩選b資料集中存在於a資料集的客戶
b_to_merge = b[b['CUST_NO'].isin(a['CUST_NO'])].copy()
print(f"從b資料集中篩選出可合併的筆數: {len(b_to_merge)}")

# 檢查a與b的CUST_NO重疊情況
a_cust_no = set(a['CUST_NO'])
b_cust_no = set(b['CUST_NO'])
overlap_count = len(a_cust_no & b_cust_no)
print(f"a與b重疊的CUST_NO數量: {overlap_count}")
print(f"a資料集中可找到對應b資料的比例: {overlap_count / len(a_cust_no) * 100:.1f}%")

# 執行左連接 (以a為主，保留a的所有記錄)
merged_step1 = a.merge(b_to_merge, on='CUST_NO', how='left')
print(f"合併b後的資料集形狀: {merged_step1.shape}")

# 檢查合併後的空值情況
b_columns = [col for col in b_to_merge.columns if col != 'CUST_NO']
print(f"來自b資料集的新欄位數: {len(b_columns)}")
print(f"新欄位中的空值統計:")
for col in b_columns[:5]:  # 顯示前5個欄位的空值情況
    null_count = merged_step1[col].isnull().sum()
    print(f"  {col}: {null_count} 個空值 ({null_count/len(merged_step1)*100:.1f}%)")
if len(b_columns) > 5:
    print(f"  ... 還有 {len(b_columns)-5} 個欄位")

print(f"\n✅ 步驟1完成：以a為基礎，成功合併b資料集")


=== 步驟1: 合併b資料集至a ===
從b資料集中篩選出可合併的筆數: 25503
a與b重疊的CUST_NO數量: 25503
a資料集中可找到對應b資料的比例: 51.1%
合併b後的資料集形狀: (49871, 74)
來自b資料集的新欄位數: 52
新欄位中的空值統計:
  MAX_SENTIMENT_SCORE: 24368 個空值 (48.9%)
  PROBLEM_COMPLEXITY_SCORE: 24368 個空值 (48.9%)
  SUB_CAT_方案異動CM轉EOC_COUNT: 24368 個空值 (48.9%)
  AVG_CALL_INTERVAL_DAYS: 24368 個空值 (48.9%)
  MAIN_CAT_非連線問題_COUNT: 24368 個空值 (48.9%)
  ... 還有 47 個欄位

✅ 步驟1完成：以a為基礎，成功合併b資料集


In [45]:
# 步驟2: 將c資料集中存在於a資料集的CUST_NO合併至已合併的資料
print("\n=== 步驟2: 合併c資料集至a ===")

# 篩選c資料集中存在於a資料集的客戶
c_to_merge = c[c['CUST_NO'].isin(a['CUST_NO'])].copy()
print(f"從c資料集中篩選出可合併的筆數: {len(c_to_merge)}")

# 執行左連接 (以step1結果為主)
final_merged = merged_step1.merge(c_to_merge, on='CUST_NO', how='left')
print(f"最終合併後的資料集形狀: {final_merged.shape}")

# 檢查合併後的空值情況
c_columns = [col for col in c_to_merge.columns if col != 'CUST_NO']
print(f"來自c資料集的新欄位數: {len(c_columns)}")
print(f"新欄位中的空值統計:")
for col in c_columns[:5]:  # 顯示前5個欄位的空值情況
    null_count = final_merged[col].isnull().sum()
    print(f"  {col}: {null_count} 個空值 ({null_count/len(final_merged)*100:.1f}%)")
if len(c_columns) > 5:
    print(f"  ... 還有 {len(c_columns)-5} 個欄位")

print(f"\n✅ 步驟2完成：成功合併c資料集至a")


=== 步驟2: 合併c資料集至a ===
從c資料集中篩選出可合併的筆數: 44968
最終合併後的資料集形狀: (49871, 89)
來自c資料集的新欄位數: 15
新欄位中的空值統計:
  平均繳款延遲日數: 4903 個空值 (9.8%)
  paytype_1: 4903 個空值 (9.8%)
  paytype_3: 4903 個空值 (9.8%)
  paytype_6: 4903 個空值 (9.8%)
  paytype_12: 4903 個空值 (9.8%)
  ... 還有 10 個欄位

✅ 步驟2完成：成功合併c資料集至a


In [46]:
# final_merged['使用狀態_數值'] = final_merged['使用狀態'].map({'使用中': 0, '未使用': 1})
final_merged['使用狀態'].value_counts()

使用狀態
使用中     42018
停用       5635
欠款斷線     1473
拆機中       316
維修中       262
設備加裝      125
暫停         19
移機中        13
派收中        10
Name: count, dtype: int64

第三版：a、b 兩資料集做聯集

In [57]:
# 新增：製作 a、b 兩資料集的聯集版本
print("=== 製作 a、b 兩資料集的聯集版本 ===")

# 執行外部連接 (outer join) - 保留所有客戶
union_merged = a.merge(b, on='CUST_NO', how='outer')

print(f"\n=== 聯集合併結果 ===")
print(f"聯集後資料集形狀: {union_merged.shape}")
print(f"聯集後客戶數: {union_merged['CUST_NO'].nunique()}")

# 檢查空值情況
print(f"\n=== 空值統計 ===")
total_nulls = union_merged.isnull().sum().sum()
total_cells = union_merged.shape[0] * union_merged.shape[1]
print(f"總空值數: {total_nulls:,}")
print(f"總空值比例: {total_nulls/total_cells*100:.2f}%")


=== 製作 a、b 兩資料集的聯集版本 ===

=== 聯集合併結果 ===
聯集後資料集形狀: (68717, 74)
聯集後客戶數: 68717

=== 空值統計 ===
總空值數: 1,662,902
總空值比例: 32.70%


In [58]:
# 步驟2: 將c資料集中存在於a資料集的CUST_NO合併至已合併的資料
print("\n=== 步驟2: 合併c資料集至聯集 ===")

# 篩選c資料集中存在於a資料集的客戶
c_to_merge = c[c['CUST_NO'].isin(union_merged['CUST_NO'])].copy()
print(f"從c資料集中篩選出可合併的筆數: {len(c_to_merge)}")

# 執行左連接 (以step1結果為主)
final_merged = union_merged.merge(c_to_merge, on='CUST_NO', how='left')
print(f"最終合併後的資料集形狀: {final_merged.shape}")

# 檢查合併後的空值情況
c_columns = [col for col in c_to_merge.columns if col != 'CUST_NO']
print(f"來自c資料集的新欄位數: {len(c_columns)}")
print(f"新欄位中的空值統計:")
for col in c_columns[:5]:  # 顯示前5個欄位的空值情況
    null_count = final_merged[col].isnull().sum()
    print(f"  {col}: {null_count} 個空值 ({null_count/len(final_merged)*100:.1f}%)")
if len(c_columns) > 5:
    print(f"  ... 還有 {len(c_columns)-5} 個欄位")

print(f"\n✅ 步驟2完成：成功合併c資料集至聯集")


=== 步驟2: 合併c資料集至聯集 ===
從c資料集中篩選出可合併的筆數: 58613
最終合併後的資料集形狀: (68717, 89)
來自c資料集的新欄位數: 15
新欄位中的空值統計:
  平均繳款延遲日數: 10104 個空值 (14.7%)
  paytype_1: 10104 個空值 (14.7%)
  paytype_3: 10104 個空值 (14.7%)
  paytype_6: 10104 個空值 (14.7%)
  paytype_12: 10104 個空值 (14.7%)
  ... 還有 10 個欄位

✅ 步驟2完成：成功合併c資料集至聯集


In [27]:
d_path = 'cleaned_dataset2.csv'                    # C
d = pd.read_csv(d_path,sep='^').rename(columns={'客編': 'CUST_NO'})

In [33]:
d.head()

,CUST_NO,產品名稱,用戶種類,相關編號,起日,迄日,系統台,地區,繳別,使用狀態
0,503,EPON,一般收視戶,2569881,2023/04/20 11:17:58,NaN,大屯,大里區,1,使用中
1,506,CATV,一般收視戶,709,1998/11/01 00:00:00,1998/11/30 00:00:00,大屯,太平區,2,停用
2,511,CATV,一般收視戶,714,2005/07/19 15:50:03,NaN,大屯,太平區,1,停用
3,511,CATV,一般收視戶,713,1997/06/01 00:00:00,1997/08/31 00:00:00,大屯,太平區,3,停用
4,511,DTV,一般收視戶,1262815,2014/06/27 19:07:39,NaN,大屯,太平區,1,停用


In [59]:
# 當final_merged資料集中使用狀態為空值時，從d資料集填補
print("=== 處理final_merged中的使用狀態空值 ===")

# 檢查final_merged中使用狀態的空值情況
if '使用狀態' in final_merged.columns:
    null_count_before = final_merged['使用狀態'].isnull().sum()
    print(f"處理前使用狀態空值數量: {null_count_before}")
    
    if null_count_before > 0:
        # 從d資料集中篩選產品名稱為EPON或CM的資料
        d_filtered = d[d['產品名稱'].isin(['EPON', 'CM'])].copy()
        print(f"d資料集中EPON/CM產品數量: {len(d_filtered)}")
        
        # 建立CUST_NO到使用狀態的對應字典
        # 如果同一個客戶有多筆記錄，取最常見的狀態
        status_mapping = d_filtered.groupby('CUST_NO')['使用狀態'].agg(
            lambda x: x.mode().iloc[0] if not x.mode().empty else x.iloc[0]
        ).to_dict()
        
        print(f"從d資料集建立的狀態對應數量: {len(status_mapping)}")
        
        # 找出final_merged中使用狀態為空值的記錄
        null_mask = final_merged['使用狀態'].isnull()
        null_cust_nos = final_merged.loc[null_mask, 'CUST_NO']
        
        print(f"需要填補的客戶數量: {len(null_cust_nos)}")
        
        # 檢查有多少客戶可以從d資料集中找到對應狀態
        available_mappings = [cust_no for cust_no in null_cust_nos if cust_no in status_mapping]
        print(f"可從d資料集找到對應狀態的客戶數量: {len(available_mappings)}")
        
        # 執行填補
        for cust_no in available_mappings:
            final_merged.loc[(final_merged['CUST_NO'] == cust_no) & 
                           (final_merged['使用狀態'].isnull()), '使用狀態'] = status_mapping[cust_no]
        
        # 檢查填補後的結果
        null_count_after = final_merged['使用狀態'].isnull().sum()
        filled_count = null_count_before - null_count_after
        
        print(f"\n=== 填補結果 ===")
        print(f"處理前空值數量: {null_count_before}")
        print(f"處理後空值數量: {null_count_after}")
        print(f"成功填補數量: {filled_count}")
        print(f"填補比例: {filled_count / null_count_before * 100:.1f}%")
        
        # 顯示填補後使用狀態的分布
        print(f"\n=== 填補後使用狀態分布 ===")
        print(final_merged['使用狀態'].value_counts(dropna=False))
        
    else:
        print("使用狀態欄位沒有空值，無需填補")
        
else:
    print("❌ final_merged中找不到'使用狀態'欄位")
    print(f"可用欄位: {list(final_merged.columns)}")

=== 處理final_merged中的使用狀態空值 ===
處理前使用狀態空值數量: 18846
d資料集中EPON/CM產品數量: 342017
從d資料集建立的狀態對應數量: 321993
需要填補的客戶數量: 18846
可從d資料集找到對應狀態的客戶數量: 9829

=== 填補結果 ===
處理前空值數量: 18846
處理後空值數量: 9017
成功填補數量: 9829
填補比例: 52.2%

=== 填補後使用狀態分布 ===
使用狀態
使用中     49104
NaN      9017
停用       7657
欠款斷線     1951
拆機中       500
維修中       282
設備加裝      140
暫停         24
移機中        20
無法完工       12
派收中        10
Name: count, dtype: int64


In [60]:
# 刪除final_merged中使用狀態為空值的記錄
print("=== 刪除使用狀態為空值的記錄 ===")

# 檢查刪除前的狀況
print(f"刪除前final_merged資料集形狀: {final_merged.shape}")

if '使用狀態' in final_merged.columns:
    # 統計空值情況
    null_count = final_merged['使用狀態'].isnull().sum()
    print(f"使用狀態空值數量: {null_count}")
    print(f"空值比例: {null_count / len(final_merged) * 100:.2f}%")
    
    if null_count > 0:
        # 顯示刪除前使用狀態的分布
        print(f"\n=== 刪除前使用狀態分布 ===")
        print(final_merged['使用狀態'].value_counts(dropna=False))
        
        # 刪除使用狀態為空值的記錄
        final_merged_cleaned = final_merged.dropna(subset=['使用狀態']).copy()
        
        print(f"\n=== 刪除結果 ===")
        print(f"刪除前筆數: {len(final_merged)}")
        print(f"刪除後筆數: {len(final_merged_cleaned)}")
        print(f"刪除筆數: {len(final_merged) - len(final_merged_cleaned)}")
        print(f"保留比例: {len(final_merged_cleaned) / len(final_merged) * 100:.2f}%")
        
        # 驗證刪除結果
        remaining_nulls = final_merged_cleaned['使用狀態'].isnull().sum()
        print(f"刪除後使用狀態空值數量: {remaining_nulls}")
        
        # 顯示刪除後使用狀態的分布
        print(f"\n=== 刪除後使用狀態分布 ===")
        print(final_merged_cleaned['使用狀態'].value_counts())
        
        # 更新final_merged變數
        final_merged = final_merged_cleaned.copy()
        
        print(f"\n✅ 已成功刪除使用狀態為空值的記錄")
        print(f"更新後final_merged形狀: {final_merged.shape}")
        
    else:
        print("✅ 使用狀態欄位沒有空值，無需刪除")
        
else:
    print("❌ final_merged中找不到'使用狀態'欄位")
    print(f"可用欄位: {list(final_merged.columns)}")

=== 刪除使用狀態為空值的記錄 ===
刪除前final_merged資料集形狀: (68717, 89)
使用狀態空值數量: 9017
空值比例: 13.12%

=== 刪除前使用狀態分布 ===
使用狀態
使用中     49104
NaN      9017
停用       7657
欠款斷線     1951
拆機中       500
維修中       282
設備加裝      140
暫停         24
移機中        20
無法完工       12
派收中        10
Name: count, dtype: int64

=== 刪除結果 ===
刪除前筆數: 68717
刪除後筆數: 59700
刪除筆數: 9017
保留比例: 86.88%
刪除後使用狀態空值數量: 0

=== 刪除後使用狀態分布 ===
使用狀態
使用中     49104
停用       7657
欠款斷線     1951
拆機中       500
維修中       282
設備加裝      140
暫停         24
移機中        20
無法完工       12
派收中        10
Name: count, dtype: int64

✅ 已成功刪除使用狀態為空值的記錄
更新後final_merged形狀: (59700, 89)


In [61]:
# 在final_merged中新增使用狀態_數值欄位
print("=== 新增使用狀態_數值欄位 ===")

# 檢查使用狀態欄位是否存在
if '使用狀態' in final_merged.columns:
    # 顯示轉換前的使用狀態分布
    print("轉換前使用狀態分布:")
    print(final_merged['使用狀態'].value_counts(dropna=False))
    
    # 創建使用狀態_數值欄位：使用中=0，其他=1
    final_merged['使用狀態_數值'] = final_merged['使用狀態'].apply(
        lambda x: 0 if x == '使用中' else 1
    )
    
    print(f"\n=== 轉換結果 ===")
    print("轉換規則: 使用中 → 0, 其他 → 1")
    
    # 顯示轉換後的數值分布
    print(f"\n使用狀態_數值分布:")
    print(final_merged['使用狀態_數值'].value_counts().sort_index())
    
    # 交叉驗證轉換結果
    print(f"\n=== 轉換驗證 ===")
    cross_tab = pd.crosstab(final_merged['使用狀態'], final_merged['使用狀態_數值'], 
                           margins=True, dropna=False)
    print("使用狀態 vs 使用狀態_數值 交叉表:")
    print(cross_tab)
    
    # 檢查是否有空值
    null_count = final_merged['使用狀態_數值'].isnull().sum()
    if null_count > 0:
        print(f"⚠️ 使用狀態_數值欄位有 {null_count} 個空值")
    else:
        print("✅ 使用狀態_數值欄位無空值")
        
    print(f"\n✅ 已成功新增使用狀態_數值欄位")
    print(f"資料集形狀: {final_merged.shape}")
    
else:
    print("❌ final_merged中找不到'使用狀態'欄位")
    print(f"可用欄位: {list(final_merged.columns)}")

=== 新增使用狀態_數值欄位 ===
轉換前使用狀態分布:
使用狀態
使用中     49104
停用       7657
欠款斷線     1951
拆機中       500
維修中       282
設備加裝      140
暫停         24
移機中        20
無法完工       12
派收中        10
Name: count, dtype: int64

=== 轉換結果 ===
轉換規則: 使用中 → 0, 其他 → 1

使用狀態_數值分布:
使用狀態_數值
0    49104
1    10596
Name: count, dtype: int64

=== 轉換驗證 ===
使用狀態 vs 使用狀態_數值 交叉表:
使用狀態_數值      0      1    All
使用狀態                        
使用中      49104      0  49104
停用           0   7657   7657
拆機中          0    500    500
暫停           0     24     24
欠款斷線         0   1951   1951
派收中          0     10     10
無法完工         0     12     12
移機中          0     20     20
維修中          0    282    282
設備加裝         0    140    140
All      49104  10596  59700
✅ 使用狀態_數值欄位無空值

✅ 已成功新增使用狀態_數值欄位
資料集形狀: (59700, 90)


In [ ]:
# 處理空值 (用0填充)
print("\n=== 處理空值 ===")
print("將所有空值填充為0...")

# 填充空值
merged_filled = final_merged.fillna(0)

# 驗證填充結果
remaining_nulls = merged_filled.isnull().sum().sum()
print(f"填充後剩餘空值數: {remaining_nulls}")

if remaining_nulls == 0:
    print("✅ 所有空值已成功填充")
else:
    print("⚠️ 仍有空值存在")

# 更新最終結果
final_dataset = merged_filled.copy()

print(f"\n=== 最終資料集 ===")
print(f"資料集形狀: {final_dataset.shape}")
print(f"空值數量: {final_dataset.isnull().sum().sum()}")


=== 處理空值 ===
將所有空值填充為0...
填充後剩餘空值數: 0
✅ 所有空值已成功填充

=== 最終資料集 ===
資料集形狀: (59700, 90)
空值數量: 0


In [72]:
# 檢查並重設索引
print("=== 檢查索引狀態 ===")

# 檢查當前索引情況
print(f"當前索引範圍: {final_dataset.index.min()} ~ {final_dataset.index.max()}")
print(f"索引長度: {len(final_dataset.index)}")
print(f"資料筆數: {len(final_dataset)}")
print(f"索引是否連續: {final_dataset.index.is_monotonic_increasing and (final_dataset.index == range(len(final_dataset))).all()}")

# 顯示前10個索引值
print(f"前10個索引值: {list(final_dataset.index[:10])}")

# 檢查是否有重複索引
duplicated_index = final_dataset.index.duplicated().sum()
print(f"重複索引數量: {duplicated_index}")

# 重設索引
print(f"\n=== 重設索引 ===")
print("正在重設索引...")

# 重設索引，讓索引從0開始連續排列
final_dataset_reindexed = final_dataset.reset_index(drop=True)

print(f"重設後索引範圍: {final_dataset_reindexed.index.min()} ~ {final_dataset_reindexed.index.max()}")
print(f"重設後索引長度: {len(final_dataset_reindexed.index)}")
print(f"重設後前10個索引值: {list(final_dataset_reindexed.index[:10])}")

# 驗證資料完整性
print(f"\n=== 驗證資料完整性 ===")
print(f"重設前資料形狀: {final_dataset.shape}")
print(f"重設後資料形狀: {final_dataset_reindexed.shape}")
print(f"資料是否一致: {final_dataset.shape == final_dataset_reindexed.shape}")

# 檢查關鍵欄位是否保持一致
if '使用狀態_數值' in final_dataset.columns:
    before_dist = final_dataset['使用狀態_數值'].value_counts().sort_index()
    after_dist = final_dataset_reindexed['使用狀態_數值'].value_counts().sort_index()
    print(f"使用狀態_數值分布是否一致: {before_dist.equals(after_dist)}")

# 更新final_dataset
final_dataset = final_dataset_reindexed.copy()

print(f"\n✅ 已成功重設索引")
print(f"最終資料集形狀: {final_dataset.shape}")
print(f"最終索引範圍: 0 ~ {len(final_dataset) - 1}")

=== 檢查索引狀態 ===
當前索引範圍: 0 ~ 68716
索引長度: 59700
資料筆數: 59700
索引是否連續: False
前10個索引值: [0, 1, 2, 6, 7, 8, 9, 10, 11, 12]
重複索引數量: 0

=== 重設索引 ===
正在重設索引...
重設後索引範圍: 0 ~ 59699
重設後索引長度: 59700
重設後前10個索引值: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

=== 驗證資料完整性 ===
重設前資料形狀: (59700, 90)
重設後資料形狀: (59700, 90)
資料是否一致: True
使用狀態_數值分布是否一致: True

✅ 已成功重設索引
最終資料集形狀: (59700, 90)
最終索引範圍: 0 ~ 59699


In [73]:
final_dataset['使用狀態_數值'].value_counts()

使用狀態_數值
0    49104
1    10596
Name: count, dtype: int64

In [74]:
final_dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 59700 entries, 0 to 59699
Data columns (total 90 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   CUST_NO                    59700 non-null  int64  
 1   工單嚴重程度_平均                  59700 non-null  float64
 2   工單嚴重程度_最高                  59700 non-null  float64
 3   工單升級趨勢                     59700 non-null  float64
 4   使用狀態                       59700 non-null  object 
 5   30天內                       59700 non-null  float64
 6   60天內                       59700 non-null  float64
 7   90天內                       59700 non-null  float64
 8   90天以上                      59700 non-null  float64
 9   平均等待天數                     59700 non-null  float64
 10  工單原因_Epon實體問題_光纖斷裂_是否發生    59700 non-null  float64
 11  工單原因_RF線路問題_調整訊號_次數        59700 non-null  float64
 12  工單原因_RF線路問題_調整訊號_佔比        59700 non-null  float64
 13  工單原因_RF線路問題_調整訊號_是否發生      59700 non-null  flo

In [75]:
# 重新排序欄位，將基本特徵排在前面
print("=== 重新排序欄位 ===")

# 檢查當前欄位
print(f"排序前final_dataset形狀: {final_dataset.shape}")
print("排序前前10個欄位:")
for i, col in enumerate(final_dataset.columns[:10], 1):
    print(f"{i:2d}. {col}")

# 定義基本特徵欄位的優先順序
basic_features = []

# 1. 客戶編號
if 'CUST_NO' in final_dataset.columns:
    basic_features.append('CUST_NO')

# 2. 使用狀態相關
if '使用狀態' in final_dataset.columns:
    basic_features.append('使用狀態')
if '使用狀態_數值' in final_dataset.columns:
    basic_features.append('使用狀態_數值')

# 3. 產品相關 (product開頭的欄位)
product_cols = [col for col in final_dataset.columns if col.startswith('product_')]
basic_features.extend(sorted(product_cols))

print(f"\n=== 識別出的基本特徵欄位 ===")
for i, col in enumerate(basic_features, 1):
    print(f"{i:2d}. {col}")

# 4. 其他重要特徵（可根據需要調整）
other_important = []
# 主要類別相關
maincat_cols = [col for col in final_dataset.columns if col.startswith('maincat_')]
subcat_cols = [col for col in final_dataset.columns if col.startswith('subcat_')]
other_important.extend(sorted(maincat_cols))
other_important.extend(sorted(subcat_cols))

# 5. 剩餘欄位
remaining_cols = [col for col in final_dataset.columns 
                 if col not in basic_features and col not in other_important]

# 建立新的欄位順序
new_column_order = basic_features + other_important + remaining_cols

print(f"\n=== 新的欄位排序 ===")
print(f"基本特徵: {len(basic_features)} 個")
print(f"其他重要特徵: {len(other_important)} 個") 
print(f"剩餘欄位: {len(remaining_cols)} 個")
print(f"總欄位數: {len(new_column_order)} 個")

=== 重新排序欄位 ===
排序前final_dataset形狀: (59700, 90)
排序前前10個欄位:
 1. CUST_NO
 2. 工單嚴重程度_平均
 3. 工單嚴重程度_最高
 4. 工單升級趨勢
 5. 使用狀態
 6. 30天內
 7. 60天內
 8. 90天內
 9. 90天以上
10. 平均等待天數

=== 識別出的基本特徵欄位 ===
 1. CUST_NO
 2. 使用狀態
 3. 使用狀態_數值
 4. product_CM
 5. product_EPON

=== 新的欄位排序 ===
基本特徵: 5 個
其他重要特徵: 0 個
剩餘欄位: 85 個
總欄位數: 90 個


In [76]:
# 執行欄位重新排序
print("\n=== 執行欄位重新排序 ===")

# 重新排序資料集
final_dataset_reordered = final_dataset[new_column_order].copy()

print(f"重新排序後的資料集形狀: {final_dataset_reordered.shape}")

# 顯示前20個欄位
print(f"\n=== 排序後前20個欄位 ===")
for i, col in enumerate(final_dataset_reordered.columns[:20], 1):
    print(f"{i:2d}. {col}")

if len(final_dataset_reordered.columns) > 20:
    print(f"... 還有 {len(final_dataset_reordered.columns) - 20} 個欄位")

# 更新final_dataset
final_dataset = final_dataset_reordered.copy()

print(f"\n✅ 已成功重新排序欄位")


=== 執行欄位重新排序 ===
重新排序後的資料集形狀: (59700, 90)

=== 排序後前20個欄位 ===
 1. CUST_NO
 2. 使用狀態
 3. 使用狀態_數值
 4. product_CM
 5. product_EPON
 6. 工單嚴重程度_平均
 7. 工單嚴重程度_最高
 8. 工單升級趨勢
 9. 30天內
10. 60天內
11. 90天內
12. 90天以上
13. 平均等待天數
14. 工單原因_Epon實體問題_光纖斷裂_是否發生
15. 工單原因_RF線路問題_調整訊號_次數
16. 工單原因_RF線路問題_調整訊號_佔比
17. 工單原因_RF線路問題_調整訊號_是否發生
18. 工單原因_Epon實體問題_光纖斷裂_佔比
19. 工單原因_Epon實體問題_光纖斷裂_次數
20. 工單原因_其他_次數
... 還有 70 個欄位

✅ 已成功重新排序欄位


In [77]:
# 驗證重新排序結果
print("\n=== 驗證重新排序結果 ===")

# 檢查基本特徵是否都在前面
print("基本特徵欄位位置:")
for col in basic_features:
    if col in final_dataset.columns:
        position = list(final_dataset.columns).index(col) + 1
        print(f"{col}: 第 {position} 位")

# 顯示前幾筆資料預覽（僅顯示前10個欄位）
print(f"\n=== 資料預覽（前10個欄位） ===")
preview_cols = final_dataset.columns[:10]
print("欄位名稱:", list(preview_cols))
print("\n前5筆資料:")
print(final_dataset[preview_cols].head())


=== 驗證重新排序結果 ===
基本特徵欄位位置:
CUST_NO: 第 1 位
使用狀態: 第 2 位
使用狀態_數值: 第 3 位
product_CM: 第 4 位
product_EPON: 第 5 位

=== 資料預覽（前10個欄位） ===
欄位名稱: ['CUST_NO', '使用狀態', '使用狀態_數值', 'product_CM', 'product_EPON', '工單嚴重程度_平均', '工單嚴重程度_最高', '工單升級趨勢', '30天內', '60天內']

前5筆資料:
   CUST_NO 使用狀態  使用狀態_數值  product_CM  product_EPON  工單嚴重程度_平均  工單嚴重程度_最高  \
0        4  使用中        0         0.0           1.0        2.0        2.0   
1       47  使用中        0         0.0           1.0        2.0        2.0   
2      138  使用中        0         0.0           1.0        2.0        2.0   
3      415  使用中        0         1.0           0.0        2.0        2.0   
4      503  使用中        0         0.0           1.0        2.0        2.0   

   工單升級趨勢  30天內  60天內  
0     0.0   1.0   0.0  
1     0.0   1.0   0.0  
2     0.0   1.0   0.0  
3     0.0   1.0   0.0  
4     0.0   1.0   0.0  


In [78]:
out_path = 'csr_service_bill_enhanced_union.csv'
final_dataset.to_csv(out_path, index=False)
print(f"✅ 資料已成功輸出至 {out_path}")

✅ 資料已成功輸出至 csr_service_bill_enhanced_union.csv
